# Association Testing

## Objective
The goal of this analysis is to investigate the relationship between Food Microbiome Exposure (FME) and gut microbiome characteristics.

Specifically, we examine whether FME is associated with:

- Alpha diversity (Shannon diversity)
- Microbiome stability
- Community composition metrics

This analysis is based on the dataset generated in the FME analysis stage.

In [63]:
#Data manipulation
import pandas as pd
import numpy as np

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Statistics
from scipy.stats import pearsonr, spearmanr
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu

#Regression
import statsmodels.api as sm
import statsmodels.formula.api as smf

#Model evaluation
import statsmodels.formula.api as smf

#Display settings
pd.set_option("display.max_columns", None)

In [64]:
from pathlib import Path
import sys
#Locate repo root (directory containing config.py) regardless of launch directory
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
DATA_DIR = _root / "data"  # robust: does not depend on __file__ inside Jupyter
sys.path.insert(0, str(_root))

## Dataset Overview

In this section, we load and inspect the statistical dataset created in the FME analysis step.

The goal is to understand the structure of the dataset, including the number of samples, available variables, data types, and missing values before performing association testing.

In [65]:
#Load the statistical dataset created in the FME analysis step

df = pd.read_csv(DATA_DIR / "fme_statistical_dataset_extended.csv")

df.head()

,fecal_sample_id,participant_id,study_day,Gender,Age,BMI,Weight,Supplement,Medications,fme_score_daily,fme_participant_mean,fme_within_person,fme_lag1,fme_lag2,fme_weighted_3day,shannon_diversity,richness,simpson_diversity,inverse_simpson,pielou_evenness,participant_shannon_cv,participant_shannon_mean,participant_n_samples,KCAL,PROT,TFAT,CARB,FIBE,SUGR,SODI,D_TOTAL,D_YOGURT,D_CHEESE,PF_MEAT,PF_SEAFD_HI,G_WHOLE
0,MCT.f.0002,MCTs01,2,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),67.196133,71.270996,-4.074863,NaN,NaN,67.196133,2.862675,211,0.871961,7.810101,0.534894,0.049204,3.066187,15,1970.043625,109.619765,53.615264,242.919015,14.419375,74.208114,4300.616500,4.156035,0.646,1.153475,0.00000,3.969,0.9718
1,MCT.f.0003,MCTs01,3,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),81.241247,71.270996,9.970251,67.196133,NaN,75.974329,3.522832,208,0.947340,18.989597,0.660011,0.049204,3.066187,15,1714.895330,89.992993,50.561373,234.225138,25.996125,67.474665,4225.813250,2.979655,0.646,1.207575,0.00000,0.000,2.9318
2,MCT.f.0004,MCTs01,4,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),126.771107,71.270996,55.500111,81.241247,67.196133,101.197154,3.031188,213,0.891830,9.244741,0.565384,0.049204,3.066187,15,2487.232625,94.611953,92.021551,257.539030,26.178425,83.960308,5329.882125,3.634460,0.000,1.482600,0.00000,0.000,0.9718
3,MCT.f.0005,MCTs01,5,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),31.484626,71.270996,-39.786370,126.771107,81.241247,70.021895,2.973836,207,0.882504,8.510902,0.557659,0.049204,3.066187,15,2260.211176,110.268353,95.570253,165.608070,13.011350,64.921966,3527.501416,3.379710,0.000,1.059000,1.81440,0.000,0.9718
4,MCT.f.0006,MCTs01,6,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),61.226577,71.270996,-10.044419,31.484626,126.771107,65.412898,3.081097,209,0.900002,10.000183,0.576732,0.049204,3.066187,15,2951.082000,161.817790,108.398516,332.637229,21.529050,119.865677,5428.573000,1.502150,0.000,0.000000,6.29596,0.000,0.4698


In [66]:
#Check that the extended FME features were loaded correctly
fme_feature_check_cols = [
    "participant_id",
    "fme_score_daily",
    "fme_participant_mean",
    "fme_within_person",
    "fme_lag1",
    "fme_lag2",
    "fme_weighted_3day"
]

df[fme_feature_check_cols].head(10)

,participant_id,fme_score_daily,fme_participant_mean,fme_within_person,fme_lag1,fme_lag2,fme_weighted_3day
0,MCTs01,67.196133,71.270996,-4.074863,NaN,NaN,67.196133
1,MCTs01,81.241247,71.270996,9.970251,67.196133,NaN,75.974329
2,MCTs01,126.771107,71.270996,55.500111,81.241247,67.196133,101.197154
3,MCTs01,31.484626,71.270996,-39.786370,126.771107,81.241247,70.021895
4,MCTs01,61.226577,71.270996,-10.044419,31.484626,126.771107,65.412898
5,MCTs01,62.023072,71.270996,-9.247925,61.226577,31.484626,55.676434
6,MCTs01,70.987405,71.270996,-0.283591,62.023072,61.226577,66.345939
7,MCTs01,54.008585,71.270996,-17.262412,70.987405,62.023072,60.705128
8,MCTs01,105.681314,71.270996,34.410318,54.008585,70.987405,83.240713
9,MCTs01,62.275914,71.270996,-8.995082,105.681314,54.008585,73.644068


In [67]:
#Check missing values in the extended FME features
df[
    [
        "fme_score_daily",
        "fme_participant_mean",
        "fme_within_person",
        "fme_lag1",
        "fme_lag2",
        "fme_weighted_3day"
    ]
].isna().sum()

fme_score_daily          0
fme_participant_mean     0
fme_within_person        0
fme_lag1                34
fme_lag2                68
fme_weighted_3day        0
dtype: int64

In [68]:
#Dataset overview

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (475, 36)

Columns:
['fecal_sample_id', 'participant_id', 'study_day', 'Gender', 'Age', 'BMI', 'Weight', 'Supplement', 'Medications', 'fme_score_daily', 'fme_participant_mean', 'fme_within_person', 'fme_lag1', 'fme_lag2', 'fme_weighted_3day', 'shannon_diversity', 'richness', 'simpson_diversity', 'inverse_simpson', 'pielou_evenness', 'participant_shannon_cv', 'participant_shannon_mean', 'participant_n_samples', 'KCAL', 'PROT', 'TFAT', 'CARB', 'FIBE', 'SUGR', 'SODI', 'D_TOTAL', 'D_YOGURT', 'D_CHEESE', 'PF_MEAT', 'PF_SEAFD_HI', 'G_WHOLE']


In [69]:
# Summary statistics

df.describe()

,study_day,Age,BMI,Weight,fme_score_daily,fme_participant_mean,fme_within_person,fme_lag1,fme_lag2,fme_weighted_3day,shannon_diversity,richness,simpson_diversity,inverse_simpson,pielou_evenness,participant_shannon_cv,participant_shannon_mean,participant_n_samples,KCAL,PROT,TFAT,CARB,FIBE,SUGR,SODI,D_TOTAL,D_YOGURT,D_CHEESE,PF_MEAT,PF_SEAFD_HI,G_WHOLE
count,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,4.750000e+02,441.000000,407.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000
mean,8.951579,31.152211,23.174737,69.069053,73.325237,73.325237,3.590111e-16,73.971508,74.886341,73.399655,2.864593,203.562105,0.859697,8.781242,0.538585,0.062621,2.864593,14.604211,2110.562806,89.144876,91.512435,229.066013,22.337148,85.219859,3491.564185,1.921608,0.110930,0.993474,1.190717,0.325082,1.263407
std,4.939697,10.156460,3.292062,14.646855,58.307131,39.803780,4.260728e+01,59.206524,60.177553,49.320072,0.413157,11.953754,0.075778,3.599882,0.074656,0.037747,0.368406,2.429422,735.450983,37.394616,40.212887,84.487698,11.600234,43.468122,1629.773797,1.541540,0.271693,1.339567,2.307983,1.576152,1.628686
min,1.000000,19.700000,17.000000,48.500000,0.000000,0.000000,-1.064991e+02,0.000000,0.000000,0.000000,1.571683,140.000000,0.464869,1.868700,0.299239,0.020213,1.944592,6.000000,722.552500,15.702908,20.467095,45.046628,3.618000,11.305484,550.853000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,23.900000,21.100000,59.100000,29.322198,53.696556,-2.508596e+01,29.569159,30.008403,38.818819,2.610804,198.000000,0.831460,5.933352,0.492687,0.036472,2.530238,14.000000,1599.179800,62.949223,60.910505,176.968102,14.850000,53.044957,2317.423250,0.664687,0.000000,0.000000,0.000000,0.000000,0.000000
50%,9.000000,28.500000,22.900000,66.900000,61.777509,62.251235,-3.085549e+00,61.600259,61.777509,64.583096,2.964396,207.000000,0.886356,8.799400,0.556193,0.048859,3.012764,15.000000,2028.168000,85.496863,85.593670,217.000000,19.693800,81.216000,3314.340000,1.650454,0.000000,0.572400,0.000000,0.000000,0.651750
75%,13.000000,37.000000,25.900000,82.900000,104.269671,99.454110,1.710716e+01,106.703015,108.547089,103.105359,3.143011,212.000000,0.910088,11.122002,0.591939,0.079944,3.130106,16.000000,2494.098500,106.259492,116.172395,271.939905,26.711850,110.042055,4310.122600,2.754320,0.000000,1.454882,1.574200,0.000000,2.093600
max,17.000000,61.600000,31.600000,104.800000,331.223987,147.889935,2.512758e+02,331.223987,331.223987,280.791849,3.731016,220.000000,0.957150,23.337217,0.693511,0.179903,3.483381,17.000000,5085.977400,256.499039,242.761818,582.015016,67.501903,261.857405,13884.247300,9.323425,1.665000,9.047425,17.577000,20.769210,9.934594


In [70]:
print(df.isnull().sum())

fecal_sample_id               0
participant_id                0
study_day                     0
Gender                        0
Age                           0
BMI                           0
Weight                        0
Supplement                    0
Medications                 255
fme_score_daily               0
fme_participant_mean          0
fme_within_person             0
fme_lag1                     34
fme_lag2                     68
fme_weighted_3day             0
shannon_diversity             0
richness                      0
simpson_diversity             0
inverse_simpson               0
pielou_evenness               0
participant_shannon_cv        0
participant_shannon_mean      0
participant_n_samples         0
KCAL                          0
PROT                          0
TFAT                          0
CARB                          0
FIBE                          0
SUGR                          0
SODI                          0
D_TOTAL                       0
D_YOGURT

In [71]:
#Define the list of alpha diversity metrics to analyze
alpha_diversity_metrics = [
    "shannon_diversity",
    "richness",
    "simpson_diversity",
    "inverse_simpson",
    "pielou_evenness"
]

alpha_diversity_metrics

['shannon_diversity',
 'richness',
 'simpson_diversity',
 'inverse_simpson',
 'pielou_evenness']

In [72]:
#Define the list of FME features to analyze
fme_features = [
    "fme_score_daily",
    "fme_participant_mean",
    "fme_within_person",
    "fme_lag1",
    "fme_lag2",
    "fme_weighted_3day"
]

fme_features

['fme_score_daily',
 'fme_participant_mean',
 'fme_within_person',
 'fme_lag1',
 'fme_lag2',
 'fme_weighted_3day']

## Correlation Analysis

In this section, we investigate the relationship between Food Microbiome Exposure (FME) and microbiome diversity metrics.

We begin by examining the correlation between the daily FME score and Shannon diversity, which is a commonly used measure of microbiome diversity.

In [73]:
#Correlation between extended FME features and alpha diversity metrics

extended_correlation_results = []

for fme_feature in fme_features:
    for metric in alpha_diversity_metrics:
        analysis_df = df[[fme_feature, metric]].dropna()

        pearson_corr, pearson_p = pearsonr(
            analysis_df[fme_feature],
            analysis_df[metric]
        )

        spearman_corr, spearman_p = spearmanr(
            analysis_df[fme_feature],
            analysis_df[metric]
        )

        extended_correlation_results.append({
            "fme_feature": fme_feature,
            "metric": metric,
            "n_samples": len(analysis_df),
            "pearson_corr": pearson_corr,
            "pearson_p_value": pearson_p,
            "spearman_corr": spearman_corr,
            "spearman_p_value": spearman_p
        })

extended_correlation_results_df = pd.DataFrame(extended_correlation_results)

display(extended_correlation_results_df.round(4))

,fme_feature,metric,n_samples,pearson_corr,pearson_p_value,spearman_corr,spearman_p_value
0,fme_score_daily,shannon_diversity,475,-0.0406,0.3776,-0.0242,0.5988
1,fme_score_daily,richness,475,0.0234,0.6113,0.0059,0.8980
2,fme_score_daily,simpson_diversity,475,-0.0407,0.3758,-0.0467,0.3101
3,fme_score_daily,inverse_simpson,475,-0.0485,0.2917,-0.0467,0.3101
4,fme_score_daily,pielou_evenness,475,-0.0439,0.3400,-0.0252,0.5834
5,fme_participant_mean,shannon_diversity,475,-0.0200,0.6642,0.0819,0.0747
6,fme_participant_mean,richness,475,-0.0081,0.8594,0.0471,0.3055
7,fme_participant_mean,simpson_diversity,475,-0.0277,0.5476,0.0289,0.5292
8,fme_participant_mean,inverse_simpson,475,-0.0650,0.1573,0.0289,0.5292
9,fme_participant_mean,pielou_evenness,475,-0.0198,0.6676,0.0803,0.0803


In [74]:
#Adjusted Models
adjusted_model_results = []

for metric in alpha_diversity_metrics:
    model_df = df[
        ["fme_score_daily", metric, "Age", "BMI"]
    ].dropna()

    formula = f"{metric} ~ fme_score_daily + Age + BMI"

    model = smf.ols(
        formula=formula,
        data=model_df
    ).fit()

    adjusted_model_results.append({
        "metric": metric,
        "model": "OLS adjusted for Age + BMI",
        "n_samples": len(model_df),
        "fme_coef": model.params.get("fme_score_daily", np.nan),
        "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
        "age_p_value": model.pvalues.get("Age", np.nan),
        "bmi_p_value": model.pvalues.get("BMI", np.nan),
        "r_squared": model.rsquared
    })

adjusted_model_results_df = pd.DataFrame(adjusted_model_results)

display(adjusted_model_results_df.round(4))

,metric,model,n_samples,fme_coef,fme_p_value,age_p_value,bmi_p_value,r_squared
0,shannon_diversity,OLS adjusted for Age + BMI,475,-0.0002,0.5154,0.2073,0.5884,0.0063
1,richness,OLS adjusted for Age + BMI,475,0.0042,0.6614,0.6000,0.8526,0.0011
2,simpson_diversity,OLS adjusted for Age + BMI,475,-0.0000,0.4961,0.1869,0.9790,0.0055
3,inverse_simpson,OLS adjusted for Age + BMI,475,-0.0021,0.4655,0.0285,0.9472,0.0127
4,pielou_evenness,OLS adjusted for Age + BMI,475,-0.0000,0.4763,0.1835,0.5967,0.0070


In [75]:
#Nutrition Covariates
nutrition_covariates = [
    "KCAL",
    "FIBE",
    "PROT",
    "TFAT",
    "CARB"
]

available_nutrition_covariates = [
    col for col in nutrition_covariates
    if col in df.columns
]

nutrition_adjusted_model_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "fme_score_daily",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = ["fme_score_daily", "Age", "BMI"] + available_nutrition_covariates
    formula = f"{metric} ~ " + " + ".join(covariates)

    model = smf.ols(
        formula=formula,
        data=model_df
    ).fit()

    nutrition_adjusted_model_results.append({
        "metric": metric,
        "model": "OLS adjusted for Age + BMI + nutrition",
        "n_samples": len(model_df),
        "fme_coef": model.params.get("fme_score_daily", np.nan),
        "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
        "age_p_value": model.pvalues.get("Age", np.nan),
        "bmi_p_value": model.pvalues.get("BMI", np.nan),
        "r_squared": model.rsquared
    })

nutrition_adjusted_model_results_df = pd.DataFrame(nutrition_adjusted_model_results)

display(nutrition_adjusted_model_results_df.round(4))

,metric,model,n_samples,fme_coef,fme_p_value,age_p_value,bmi_p_value,r_squared
0,shannon_diversity,OLS adjusted for Age + BMI + nutrition,475,0.0000,0.9498,0.0421,0.5993,0.0354
1,richness,OLS adjusted for Age + BMI + nutrition,475,0.0223,0.0596,0.9104,0.8926,0.0297
2,simpson_diversity,OLS adjusted for Age + BMI + nutrition,475,-0.0000,0.7941,0.0361,0.2598,0.0351
3,inverse_simpson,OLS adjusted for Age + BMI + nutrition,475,0.0004,0.9054,0.0064,0.3817,0.0367
4,pielou_evenness,OLS adjusted for Age + BMI + nutrition,475,-0.0000,0.9410,0.0352,0.5747,0.0359


In [76]:
#Mixed Models with Participant Random Intercept
mixed_model_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "participant_id",
        "fme_score_daily",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = ["fme_score_daily", "Age", "BMI"] + available_nutrition_covariates
    formula = f"{metric} ~ " + " + ".join(covariates)

    try:
        model = smf.mixedlm(
            formula=formula,
            data=model_df,
            groups=model_df["participant_id"]
        ).fit()

        mixed_model_results.append({
            "metric": metric,
            "model": "MixedLM adjusted for Age + BMI + nutrition + participant random intercept",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),
            "fme_coef": model.params.get("fme_score_daily", np.nan),
            "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
            "age_p_value": model.pvalues.get("Age", np.nan),
            "bmi_p_value": model.pvalues.get("BMI", np.nan)
        })

    except Exception as e:
        mixed_model_results.append({
            "metric": metric,
            "model": "MixedLM failed",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),
            "fme_coef": np.nan,
            "fme_p_value": np.nan,
            "age_p_value": np.nan,
            "bmi_p_value": np.nan,
            "error": str(e)
        })

mixed_model_results_df = pd.DataFrame(mixed_model_results)

display(mixed_model_results_df.round(4))

c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,metric,model,n_samples,n_participants,fme_coef,fme_p_value,age_p_value,bmi_p_value
0,shannon_diversity,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0004,0.0510,0.6997,0.8906
1,richness,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,0.0072,0.4767,0.8633,0.9619
2,simpson_diversity,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0001,0.2745,0.6413,0.9563
3,inverse_simpson,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0010,0.6792,0.5421,0.8908
4,pielou_evenness,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0001,0.0392,0.6797,0.8965


In [77]:
#Mixed models separating between-person and within-person FME effects

mixed_model_decomposed_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "participant_id",
        "fme_participant_mean",
        "fme_within_person",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = [
        "fme_participant_mean",
        "fme_within_person",
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    formula = f"{metric} ~ " + " + ".join(covariates)

    try:
        model = smf.mixedlm(
            formula=formula,
            data=model_df,
            groups=model_df["participant_id"]
        ).fit()

        mixed_model_decomposed_results.append({
            "metric": metric,
            "model": "MixedLM decomposed FME: participant mean + within-person",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_participant_mean_coef": model.params.get("fme_participant_mean", np.nan),
            "fme_participant_mean_p_value": model.pvalues.get("fme_participant_mean", np.nan),

            "fme_within_person_coef": model.params.get("fme_within_person", np.nan),
            "fme_within_person_p_value": model.pvalues.get("fme_within_person", np.nan),

            "age_p_value": model.pvalues.get("Age", np.nan),
            "bmi_p_value": model.pvalues.get("BMI", np.nan)
        })

    except Exception as e:
        mixed_model_decomposed_results.append({
            "metric": metric,
            "model": "MixedLM decomposed FME failed",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_participant_mean_coef": np.nan,
            "fme_participant_mean_p_value": np.nan,

            "fme_within_person_coef": np.nan,
            "fme_within_person_p_value": np.nan,

            "age_p_value": np.nan,
            "bmi_p_value": np.nan,
            "error": str(e)
        })

mixed_model_decomposed_results_df = pd.DataFrame(mixed_model_decomposed_results)

display(mixed_model_decomposed_results_df.round(4))

c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,metric,model,n_samples,n_participants,fme_participant_mean_coef,fme_participant_mean_p_value,fme_within_person_coef,fme_within_person_p_value,age_p_value,bmi_p_value
0,shannon_diversity,MixedLM decomposed FME: participant mean + wit...,475,34,-0.0002,0.8946,-0.0005,0.0506,0.6937,0.8879
1,richness,MixedLM decomposed FME: participant mean + wit...,475,34,-0.0145,0.7389,0.0079,0.4379,0.7869,0.9829
2,simpson_diversity,MixedLM decomposed FME: participant mean + wit...,475,34,-0.0000,0.9447,-0.0001,0.2726,0.6400,0.9612
3,inverse_simpson,MixedLM decomposed FME: participant mean + wit...,475,34,-0.0052,0.7407,-0.0009,0.6989,0.5942,0.9025
4,pielou_evenness,MixedLM decomposed FME: participant mean + wit...,475,34,-0.0000,0.9082,-0.0001,0.0386,0.6685,0.8920


In [78]:
#Mixed models using weighted recent 3 day FME exposure

mixed_model_weighted_3day_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "participant_id",
        "fme_weighted_3day",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = [
        "fme_weighted_3day",
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    formula = f"{metric} ~ " + " + ".join(covariates)

    try:
        model = smf.mixedlm(
            formula=formula,
            data=model_df,
            groups=model_df["participant_id"]
        ).fit()

        mixed_model_weighted_3day_results.append({
            "metric": metric,
            "model": "MixedLM weighted 3-day FME",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_weighted_3day_coef": model.params.get("fme_weighted_3day", np.nan),
            "fme_weighted_3day_p_value": model.pvalues.get("fme_weighted_3day", np.nan),

            "age_p_value": model.pvalues.get("Age", np.nan),
            "bmi_p_value": model.pvalues.get("BMI", np.nan)
        })

    except Exception as e:
        mixed_model_weighted_3day_results.append({
            "metric": metric,
            "model": "MixedLM weighted 3-day FME failed",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_weighted_3day_coef": np.nan,
            "fme_weighted_3day_p_value": np.nan,

            "age_p_value": np.nan,
            "bmi_p_value": np.nan,
            "error": str(e)
        })

mixed_model_weighted_3day_results_df = pd.DataFrame(mixed_model_weighted_3day_results)

display(mixed_model_weighted_3day_results_df.round(4))

c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,metric,model,n_samples,n_participants,fme_weighted_3day_coef,fme_weighted_3day_p_value,age_p_value,bmi_p_value
0,shannon_diversity,MixedLM weighted 3-day FME,475,34,-0.0006,0.0595,0.7183,0.8982
1,richness,MixedLM weighted 3-day FME,475,34,0.0088,0.5485,0.8729,0.9595
2,simpson_diversity,MixedLM weighted 3-day FME,475,34,-0.0001,0.1904,0.6603,0.9466
3,inverse_simpson,MixedLM weighted 3-day FME,475,34,-0.0020,0.5640,0.5509,0.8962
4,pielou_evenness,MixedLM weighted 3-day FME,475,34,-0.0001,0.0487,0.6995,0.9045


In [79]:
#Mixed models using lagged FME exposure features

mixed_model_lagged_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "participant_id",
        "fme_lag1",
        "fme_lag2",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = [
        "fme_lag1",
        "fme_lag2",
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    formula = f"{metric} ~ " + " + ".join(covariates)

    try:
        model = smf.mixedlm(
            formula=formula,
            data=model_df,
            groups=model_df["participant_id"]
        ).fit()

        mixed_model_lagged_results.append({
            "metric": metric,
            "model": "MixedLM lagged FME: lag1 + lag2",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_lag1_coef": model.params.get("fme_lag1", np.nan),
            "fme_lag1_p_value": model.pvalues.get("fme_lag1", np.nan),

            "fme_lag2_coef": model.params.get("fme_lag2", np.nan),
            "fme_lag2_p_value": model.pvalues.get("fme_lag2", np.nan),

            "age_p_value": model.pvalues.get("Age", np.nan),
            "bmi_p_value": model.pvalues.get("BMI", np.nan)
        })

    except Exception as e:
        mixed_model_lagged_results.append({
            "metric": metric,
            "model": "MixedLM lagged FME failed",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),

            "fme_lag1_coef": np.nan,
            "fme_lag1_p_value": np.nan,

            "fme_lag2_coef": np.nan,
            "fme_lag2_p_value": np.nan,

            "age_p_value": np.nan,
            "bmi_p_value": np.nan,
            "error": str(e)
        })

mixed_model_lagged_results_df = pd.DataFrame(mixed_model_lagged_results)

display(mixed_model_lagged_results_df.round(4))

c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,metric,model,n_samples,n_participants,fme_lag1_coef,fme_lag1_p_value,fme_lag2_coef,fme_lag2_p_value,age_p_value,bmi_p_value
0,shannon_diversity,MixedLM lagged FME: lag1 + lag2,407,34,0.0000,0.9285,-0.0004,0.1126,0.6912,0.8931
1,richness,MixedLM lagged FME: lag1 + lag2,407,34,0.0078,0.4233,-0.0111,0.2570,0.8304,0.8672
2,simpson_diversity,MixedLM lagged FME: lag1 + lag2,407,34,-0.0000,0.7483,-0.0000,0.3571,0.6373,0.9570
3,inverse_simpson,MixedLM lagged FME: lag1 + lag2,407,34,-0.0010,0.6739,-0.0005,0.8232,0.5226,0.9628
4,pielou_evenness,MixedLM lagged FME: lag1 + lag2,407,34,0.0000,0.9821,-0.0001,0.1367,0.6698,0.9051


In [80]:
#Final summary table across all association models

summary_tables = []

#Original correlation results: fme_score_daily only
correlation_summary = correlation_results_df.copy()
correlation_summary["model_type"] = "Correlation"
correlation_summary["fme_feature"] = "fme_score_daily"
correlation_summary["fme_coef"] = correlation_summary["pearson_corr"]
correlation_summary["fme_p_value"] = correlation_summary["pearson_p_value"]
correlation_summary = correlation_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(correlation_summary)

#Extended correlation results: all FME features
extended_correlation_summary = extended_correlation_results_df.copy()
extended_correlation_summary["model_type"] = "Extended correlation"
extended_correlation_summary["fme_coef"] = extended_correlation_summary["pearson_corr"]
extended_correlation_summary["fme_p_value"] = extended_correlation_summary["pearson_p_value"]
extended_correlation_summary = extended_correlation_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(extended_correlation_summary)

#OLS adjusted for Age + BMI: fme_score_daily only
ols_age_bmi_summary = adjusted_model_results_df.copy()
ols_age_bmi_summary["model_type"] = "OLS: Age + BMI"
ols_age_bmi_summary["fme_feature"] = "fme_score_daily"
ols_age_bmi_summary = ols_age_bmi_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(ols_age_bmi_summary)

#OLS adjusted for Age + BMI + nutrition: fme_score_daily only
ols_nutrition_summary = nutrition_adjusted_model_results_df.copy()
ols_nutrition_summary["model_type"] = "OLS: Age + BMI + nutrition"
ols_nutrition_summary["fme_feature"] = "fme_score_daily"
ols_nutrition_summary = ols_nutrition_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(ols_nutrition_summary)

#Mixed effects model: fme_score_daily only
mixed_summary = mixed_model_results_df.copy()
mixed_summary["model_type"] = "MixedLM: Age + BMI + nutrition + participant"
mixed_summary["fme_feature"] = "fme_score_daily"
mixed_summary = mixed_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(mixed_summary)

#MixedLM decomposed model: participant mean and within-person FME
decomposed_summary_rows = []

for _, row in mixed_model_decomposed_results_df.iterrows():
    decomposed_summary_rows.append({
        "metric": row["metric"],
        "model_type": "MixedLM: decomposed FME + Age + BMI + nutrition + participant",
        "fme_feature": "fme_participant_mean",
        "n_samples": row["n_samples"],
        "fme_coef": row["fme_participant_mean_coef"],
        "fme_p_value": row["fme_participant_mean_p_value"]
    })

    decomposed_summary_rows.append({
        "metric": row["metric"],
        "model_type": "MixedLM: decomposed FME + Age + BMI + nutrition + participant",
        "fme_feature": "fme_within_person",
        "n_samples": row["n_samples"],
        "fme_coef": row["fme_within_person_coef"],
        "fme_p_value": row["fme_within_person_p_value"]
    })

decomposed_summary = pd.DataFrame(decomposed_summary_rows)
summary_tables.append(decomposed_summary)

#MixedLM weighted 3-day FME model
weighted_3day_summary = mixed_model_weighted_3day_results_df.copy()
weighted_3day_summary["model_type"] = "MixedLM: weighted 3-day FME + Age + BMI + nutrition + participant"
weighted_3day_summary["fme_feature"] = "fme_weighted_3day"
weighted_3day_summary["fme_coef"] = weighted_3day_summary["fme_weighted_3day_coef"]
weighted_3day_summary["fme_p_value"] = weighted_3day_summary["fme_weighted_3day_p_value"]
weighted_3day_summary = weighted_3day_summary[
    ["metric", "model_type", "fme_feature", "n_samples", "fme_coef", "fme_p_value"]
]
summary_tables.append(weighted_3day_summary)

#MixedLM lagged FME model: lag1 and lag2
lagged_summary_rows = []

for _, row in mixed_model_lagged_results_df.iterrows():
    lagged_summary_rows.append({
        "metric": row["metric"],
        "model_type": "MixedLM: lagged FME + Age + BMI + nutrition + participant",
        "fme_feature": "fme_lag1",
        "n_samples": row["n_samples"],
        "fme_coef": row["fme_lag1_coef"],
        "fme_p_value": row["fme_lag1_p_value"]
    })

    lagged_summary_rows.append({
        "metric": row["metric"],
        "model_type": "MixedLM: lagged FME + Age + BMI + nutrition + participant",
        "fme_feature": "fme_lag2",
        "n_samples": row["n_samples"],
        "fme_coef": row["fme_lag2_coef"],
        "fme_p_value": row["fme_lag2_p_value"]
    })

lagged_summary = pd.DataFrame(lagged_summary_rows)
summary_tables.append(lagged_summary)

#Combine all summaries
final_association_summary = pd.concat(
    summary_tables,
    axis=0,
    ignore_index=True
)

#Add significance label
final_association_summary["significance"] = np.where(
    final_association_summary["fme_p_value"] < 0.05,
    "significant",
    np.where(
        final_association_summary["fme_p_value"] < 0.10,
        "trend",
        "not significant"
    )
)

#Sort for readability
final_association_summary = final_association_summary.sort_values(
    by=["metric", "model_type", "fme_feature"]
).reset_index(drop=True)

display(final_association_summary.round(4))

,metric,model_type,fme_feature,n_samples,fme_coef,fme_p_value,significance
0,inverse_simpson,Correlation,fme_score_daily,475,-0.0485,0.2917,not significant
1,inverse_simpson,Extended correlation,fme_lag1,441,-0.0641,0.1788,not significant
2,inverse_simpson,Extended correlation,fme_lag2,407,-0.0672,0.1758,not significant
3,inverse_simpson,Extended correlation,fme_participant_mean,475,-0.0650,0.1573,not significant
4,inverse_simpson,Extended correlation,fme_score_daily,475,-0.0485,0.2917,not significant
...,...,...,...,...,...,...,...
70,simpson_diversity,MixedLM: lagged FME + Age + BMI + nutrition + ...,fme_lag1,407,-0.0000,0.7483,not significant
71,simpson_diversity,MixedLM: lagged FME + Age + BMI + nutrition + ...,fme_lag2,407,-0.0000,0.3571,not significant
72,simpson_diversity,MixedLM: weighted 3-day FME + Age + BMI + nutr...,fme_weighted_3day,475,-0.0001,0.1904,not significant
73,simpson_diversity,OLS: Age + BMI,fme_score_daily,475,-0.0000,0.4961,not significant


## Summary of Association Testing Results

In this analysis, we tested associations between FME and gut microbiome alpha diversity using several FME representations and multiple statistical models.

The alpha diversity outcomes included:

* Shannon diversity
* Richness
* Simpson diversity
* Inverse Simpson
* Pielou evenness

The FME predictors included:

* `fme_score_daily` — the original daily weighted FME score
* `fme_participant_mean` — the participant-level mean FME score
* `fme_within_person` — deviation of each daily FME score from that participant’s mean
* `fme_lag1` — previous FME score for the same participant
* `fme_lag2` — FME score two previous observations back for the same participant
* `fme_weighted_3day` — weighted recent FME exposure based on the current score, lag 1, and lag 2

First, simple Pearson and Spearman correlations were calculated between the extended FME features and each alpha diversity metric. These unadjusted correlations did not show strong or consistent significant associations.

Next, linear regression models adjusted for Age and BMI were fitted for each diversity metric using the daily FME score. In these models, FME was not significantly associated with any of the alpha diversity metrics.

We then extended the regression models by additionally adjusting for nutritional covariates, including total calorie intake, fiber, protein, fat, and carbohydrates. In this model set, richness showed a near-significant trend with daily FME, but did not pass the conventional significance threshold of 0.05.

Because the dataset contains repeated fecal samples from the same participants, we fitted mixed effects models with participant-specific random intercepts. These models adjusted for Age, BMI, nutritional covariates, and repeated sampling within participants.

In the original mixed effects model using `fme_score_daily`, Pielou evenness showed a statistically significant association with FME. Shannon diversity showed a near-significant trend. Both associations had negative coefficients, suggesting that higher FME scores may be associated with lower microbial evenness and lower Shannon diversity.

We then fitted decomposed mixed effects models separating between-person and within-person FME effects. In these models, the within-person FME component was associated with Pielou evenness and showed a near-significant trend with Shannon diversity. The participant-level mean FME component was not significant. This suggests that the observed association may be driven more by day-to-day deviations within the same participant than by stable differences between participants.

We also tested short-term exposure models using `fme_weighted_3day`, which combines the current FME score with the two previous observations. In this model, Pielou evenness was significantly associated with weighted recent FME, and Shannon diversity showed a near-significant trend.

Finally, we tested lagged models using `fme_lag1` and `fme_lag2` separately. These lagged predictors did not show significant associations when included as separate predictors, suggesting that the combined recent exposure score may capture the short-term signal better than individual lag terms.

Overall, the results suggest that the association between FME and gut microbiome alpha diversity is not strongly reflected in simple unadjusted correlations or standard OLS models. However, after accounting for repeated samples per participant using mixed effects models, FME appears to be associated specifically with microbial evenness, and possibly Shannon diversity. The strongest and most consistent signals appear in the within-person FME model and the weighted recent FME exposure model, rather than in richness or dominance-based diversity metrics.


In [81]:
#Save final association summary table

final_association_summary.to_csv(
    DATA_DIR / "association_testing_extended_summary.csv",
    index=False
)

print("Saved association_testing_extended_summary.csv")

Saved association_testing_extended_summary.csv
